In [0]:
# statsmodels (GLM fitting) and openpyxl (Excel export) — fast no-ops if already present
%pip install statsmodels openpyxl

# Tweedie GLM — motor, buildings & contents

Classic actuarial GLM counterpart to the Tweedie XGBoost models (Motor/Buildings/Contents_Bridget), built 2026-10-21. The XGBoost results are used to select the rating factors; the master rebuild, claims development, cross-validation and calibration structure are IDENTICAL to the XGBoost notebooks so the two model families are directly comparable.

## Statistical model & distributions assumed

* **Response**: `ultimate_claims_dev` — claims net of recoveries **developed to ultimate** per policy-year (open-claim tail factors by peril class + IBNR; identical assumptions to the XGBoost notebooks, 2026-10-21, see AI_ASSISTANCE.md).
* **Distribution**: Tweedie **compound Poisson–Gamma** (1 < p < 2). Claim count N ~ Poisson(λ·exposure); individual claim severities Zᵢ ~ Gamma(α, θ) i.i.d.; Y = Σ Zᵢ with a point mass at 0 for claim-free policy-years. Var(Y) = φ·μ^p.
* **Variance power p = 1.5 FIXED** — matches the hackathon scoring (mean Tweedie deviance D² at p = 1.5). Not estimated from data (HACKATHON-SCORING DECISION).
* **Link**: log. E[Y] = exposure · exp(x′β), fitted as a GLM with **offset log(exposure)** and **variance weights = exposure** — the standard aggregated pure-premium structure. Exposure enters through the offset and the weights, NOT as a feature (unlike the XGBoost, which used it both as feature and weight).
* **Dispersion φ**: estimated (Pearson χ²/df); standard errors are asymptotic and φ-scaled.
* **Implementation**: hand-rolled Fisher-scoring IRLS, chunked — the IDENTICAL estimator `statsmodels` GLM produces, but its built-in IRLS OOM-killed the driver on the 980k×67 motor design (exit 137). SEs from φ(X′WX)⁻¹ at convergence; p-values normal-approx; design stored float32.

## Assumptions (full list)

1. **Claims development**: closed/repudiated = ultimate (factor 1.0); OPEN claims × benchmark tail factors by peril class & loss year (long: bodily_injury/subsidence ×1.10–1.40; medium: fire/flood ×1.02–1.15; short: rest ×1.00–1.05); IBNR gross-up from the 2021–22 reporting-lag distribution (motor 2024 ×1.033, buildings ×1.017, contents ×1.007). Single 31/12/2024 snapshot — factors are ASSUMPTIONS (AI_ASSISTANCE.md 2026-10-21).
2. **Population**: ALL policies (no renewal filter — 2026-10-21 human decision); exposure recomputed from period dates per line (same rules as the XGBoost notebooks: motor/contents days/365, buildings leap-adjusted & capped at 1.0).
3. **Feature selection (XGBoost insight)**: top-15 features by XGBoost gain per line from the 2026-10-21 rate-model runs, MINUS `exposure` (14 rating factors per line; gain shares in the Excel results workbooks). Rationale: the GLM gets the factors the trees verified as most predictive; kept parsimonious for interpretability and runtime.
4. **Functional form**: numeric effects are LINEAR on the log scale (no binning/splines) — a simplification vs the trees' interactions and non-linearity; the CV comparison quantifies what it costs.
5. **Numerics standardized** to train z-scores — purely for numerical conditioning of IRLS; does not change the model space (affine reparameterisation absorbed by the intercept).
6. **Categoricals**: one-hot with the training MODE level as reference (coefficients are relative to the most common level); levels beyond the top-29 by training frequency bucketed to `OTHER` (bounds design width — matters for `make_model`/`occupation`).
7. **Nulls**: numeric → train median, categorical → train mode (same philosophy as the XGB preprocessing); a `<col>_isna` missing indicator is added for features with ≥5% training nulls (mostly the ~10% customer-join gaps).
8. **Cross-validation**: 4-fold GroupKFold grouped by customer_id (same structure as XGBoost) — honest OOF; evaluation with mean Tweedie unit deviance at p=1.5 on the claims scale; final model refit on 100% of rows.
9. **Calibration**: per-line multiplicative scaling to the DEVELOPED training row-mean (same CAL_TARGET as v012) at the combined assembly step. Note: an intercept + log link + exposure offset makes the GLM match the exposure-weighted claims total EXACTLY on training data, so the GLM's level gap vs the XGBoost's 0.5–0.66× is expected to be much smaller — calibration factors near 1.
10. Zero-exposure rows would break the log offset — asserted none exist (verified in the XGB runs).

## Outputs

* Per-line CV fold tables, OOF decile checks and coefficient tables (coef, exp(coef) multiplicative factor, std err, p-value) vs the XGBoost references (deviance: motor 135.32, buildings 140.13, contents 70.97; top-decile lift 2.30×/2.79×/2.64×).
* Per-line GLM submission CSVs (uncalibrated model output, dummy 0.01 for other lines) and the calibrated combined `motor_buildings_contents_submission_v013_glm.csv`.
* `Excel files/glm_tweedie_results.xlsx` with everything for review.

#### Load data

In [0]:
import os

# Same loader as the per-line XGBoost notebooks: every parquet in the repo's data/ folder
data_dir = "/Workspace/Users/bridget.macdonnell1@gmail.com/hackathon-project/data"
dataframes = {}
for file_name in sorted(f for f in os.listdir(data_dir) if f.endswith(".parquet")):
    name = file_name.replace(".parquet", "")
    dataframes[name] = spark.read.parquet(os.path.join(data_dir, file_name))
print(f"Loaded {len(dataframes)} datasets: {', '.join(sorted(dataframes))}")

#### Shared functions — master rebuild with claims development, GLM design matrix, fitting & evaluation

Everything below is parameterised by line so motor/buildings/contents run the identical pipeline. The rebuild and development logic is byte-for-byte the same set of assumptions as the XGBoost notebooks' rebuild cells (2026-10-21, AI_ASSISTANCE.md).

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import numpy as np
import pandas as pd
import gc
import statsmodels.api as sm
from sklearn.model_selection import GroupKFold
from scipy.stats import norm as _norm_dist

# ---- global model configuration (see the spec cell for the full assumption list) ----
POWER = 1.5                    # Tweedie variance power, FIXED to match the hackathon scoring (D2 at p=1.5)
N_FOLDS = 4                    # same GroupKFold structure as the XGBoost models
MAX_LEVELS = 30                # categoricals wider than this keep the top-29 train levels + OTHER
MISSING_INDICATOR_MIN = 0.05   # features with >=5% train nulls also get a <col>_isna indicator
DUMMY = 0.01                   # no-model rows in per-line submission files (scorer rejects zeros)
TARGET = "ultimate_claims_dev"
GROUP_COL = "customer_id"

# Open-claim tail factors by peril class & loss year, and IBNR gross-ups — identical
# assumptions to the XGBoost notebooks' rebuild cells (2026-10-21; AI_ASSISTANCE.md).
OPEN_TAIL = {
    "long":   {2021: 1.10, 2022: 1.15, 2023: 1.25, 2024: 1.40},   # bodily_injury, subsidence
    "medium": {2021: 1.02, 2022: 1.04, 2023: 1.08, 2024: 1.15},   # fire, flood
    "short":  {2021: 1.00, 2022: 1.00, 2023: 1.02, 2024: 1.05},    # all other perils
}
IBNR = {
    "motor":     {2021: 1.000, 2022: 1.000, 2023: 1.004, 2024: 1.033},
    "buildings": {2021: 1.000, 2022: 1.000, 2023: 1.001, 2024: 1.017},
    "contents":  {2021: 1.000, 2022: 1.000, 2023: 1.000, 2024: 1.007},
}

# GLM rating factors: top-15 features by XGBoost gain per line from the 2026-10-21
# rate-model runs, with `exposure` EXCLUDED (it enters the GLM as offset + variance
# weight, not as a feature). Gain shares recorded in the Excel results workbooks.
# Year added as first feature (2026-10-22): numeric trend on the log scale — the GLM
# coefficient gives a multiplicative annual trend (exp(beta) = 1.05 means +5%/yr).
# This lets the GLM extrapolate the trend to 2025, which the XGBoost (trees) cannot.
LINE_CONFIG = {
    "motor": dict(features=[
        "Year",
        "ncb_years", "make_model", "years_licensed", "driver_age", "prior_claims_5y",
        "cover_type", "payment_method", "voluntary_excess", "annual_mileage",
        "vehicle_use", "channel", "missed_payments_12m", "urban_density", "occupation"]),
    "buildings": dict(features=[
        "Year",
        "flood_zone", "elevation_m", "sum_insured", "prior_claims_5y", "missed_payments_12m",
        "occupancy", "bedrooms", "occupation", "payment_method", "distance_to_river_km",
        "region", "year_built", "basement", "channel"]),
    "contents": dict(features=[
        "Year",
        "flood_zone", "sum_insured", "elevation_m", "prior_claims_5y", "voluntary_excess",
        "accidental_damage_cover", "new_business", "tenure", "occupation",
        "missed_payments_12m", "payment_method", "region", "urban_density",
        "distance_to_river_km"]),
}

# XGBoost reference results (2026-10-21 rate-model runs) for direct comparison
XGB_REF = {
    "motor":     dict(deviance=135.32, monotonicity=0.89, lift=2.30),
    "buildings": dict(deviance=140.13, monotonicity=1.00, lift=2.79),
    "contents":  dict(deviance=70.97,  monotonicity=1.00, lift=2.64),
}


def rebuild_master(line):
    """ALL-policies master for one line, claims developed to ultimate — identical joins,
    tail factors and IBNR to the XGBoost notebooks' rebuild cells (no renewal filter)."""
    _tail_col = (F.when(F.col("peril").isin("bodily_injury", "subsidence"), F.lit("long"))
                  .when(F.col("peril").isin("fire", "flood"), F.lit("medium"))
                  .otherwise(F.lit("short")))
    _open = None
    for _t, _yrs in OPEN_TAIL.items():
        for _yr, _f in _yrs.items():
            _c = (_tail_col == F.lit(_t)) & (F.col("_claim_year") == F.lit(_yr))
            _open = F.when(_c, F.lit(_f)) if _open is None else _open.when(_c, F.lit(_f))
    _open = _open.otherwise(F.lit(1.0))
    _ibnr = None
    for _yr, _f in IBNR[line].items():
        _c = F.col("_claim_year") == F.lit(_yr)
        _ibnr = F.when(_c, F.lit(_f)) if _ibnr is None else _ibnr.when(_c, F.lit(_f))
    _ibnr = _ibnr.otherwise(F.lit(1.0))

    wx = (dataframes["weather"]
          .withColumnRenamed("address_id", "_wx_address_id")
          .withColumnRenamed("year", "_wx_year"))
    cl = (dataframes["claims"]
          .filter(F.col("line") == line)
          .withColumn("_claim_year", F.year("loss_date"))
          .withColumn("_dev_amount",
                      (F.col("incurred_to_date") - F.col("recoveries"))
                      * F.when(F.col("status") != "open", F.lit(1.0)).otherwise(_open)
                      * _ibnr)
          .withColumn("_rn", F.row_number().over(
              Window.partitionBy("policy_id", "_claim_year").orderBy("claim_id"))))
    cl_agg = (cl.groupBy("policy_id", "_claim_year")
              .agg(F.sum("incurred_to_date").alias("incurred_to_date"),
                   F.sum("recoveries").alias("recoveries"),
                   F.count(F.lit(1)).alias("n_claims"),
                   F.round(F.sum("_dev_amount"), 2).alias(TARGET))
              .withColumnRenamed("policy_id", "_cl_policy_id"))
    return (dataframes[f"policies_{line}"]
        .join(wx, (F.col("address_id") == F.col("_wx_address_id")) &
                  (F.year(F.col("period_start")) == F.col("_wx_year")), "left")
        .drop("_wx_address_id", "_wx_year")
        .join(dataframes["customers"], "customer_id", "left")
        .join(dataframes["addresses"], "address_id", "left")
        .join(cl_agg, (F.col("policy_id") == F.col("_cl_policy_id")) &
                      (F.year(F.col("period_start")) == F.col("_claim_year")), "left")
        .drop("_cl_policy_id", "_claim_year"))


def exposure_expr(line):
    """Same exposure recompute as each line's XGBoost notebook (2026-10-21): motor and
    contents days-in-force/365; buildings leap-adjusted and capped at 1.0."""
    if line == "buildings":
        return F.least((F.datediff("period_end", "period_start") + F.lit(1)) /
                       F.when(F.year("period_start") % 4 == 0, F.lit(366)).otherwise(F.lit(365)),
                       F.lit(1.0))
    return (F.datediff("period_end", "period_start") + F.lit(1)) / F.lit(365)


def modeling_pandas(line, master):
    feats = LINE_CONFIG[line]["features"]
    mdf = (master
        .withColumn("exposure", exposure_expr(line))
        .withColumn(TARGET, F.coalesce(F.col(TARGET), F.lit(0.0)))
        .select("row_id", "policy_id", GROUP_COL, "exposure", *feats, TARGET))
    missing = [c for c in feats if c not in mdf.columns]
    assert not missing, f"{line}: features missing from the master: {missing}"
    return mdf.toPandas()


def test_pandas(line):
    """2025 test master for one line, mirroring the training joins (no claims join)."""
    feats = LINE_CONFIG[line]["features"]
    wx = (dataframes["weather"]
          .withColumnRenamed("address_id", "_wx_address_id")
          .withColumnRenamed("year", "_wx_year"))
    tdf = (dataframes[f"test_{line}"]
        .join(wx, (F.col("address_id") == F.col("_wx_address_id")) &
                  (F.year(F.col("period_start")) == F.col("_wx_year")), "left")
        .drop("_wx_address_id", "_wx_year")
        .join(dataframes["customers"], "customer_id", "left")
        .join(dataframes["addresses"], "address_id", "left")
        .withColumn("exposure", exposure_expr(line))
        .select("row_id", "policy_id", GROUP_COL, "exposure", *feats))
    missing = [c for c in feats if c not in tdf.columns]
    assert not missing, f"{line}: features missing from the test master: {missing}"
    return tdf.toPandas()


def _normalize(pdf, feats):
    """bool -> float (0/1/NaN); object-typed bool columns -> float. Returns a copy."""
    df = pdf[feats].copy()
    for c in feats:
        if df[c].dtype == bool:
            df[c] = df[c].astype("float")
        elif df[c].dtype == object:
            nn = df[c].dropna()
            if len(nn) and set(map(type, nn.unique())) <= {bool}:
                df[c] = df[c].map(lambda v: float(v) if isinstance(v, (bool, np.bool_)) else np.nan)
    return df


def build_design_train(pdf, feats):
    """TRAIN design matrix: median/mode imputation, missing indicators (>=5% nulls),
    cardinality cap (top-29 + OTHER), one-hot with the training MODE level as reference,
    z-standardized numerics. Returns (X, info); info replays everything on test."""
    df = _normalize(pdf, feats)
    info = {"features": list(feats), "impute": {}, "ind_cols": [], "num_stats": {},
            "mode_ref": {}, "keep_levels": {}, "dummy_cols": [], "design_cols": []}
    for c in feats:
        na = df[c].isna()
        if na.any():
            fill = df[c].mode(dropna=True).iloc[0] if df[c].dtype == object else float(df[c].median())
            df[c] = df[c].fillna(fill)
            info["impute"][c] = fill
            if na.mean() >= MISSING_INDICATOR_MIN:
                df[c + "_isna"] = na.to_numpy(dtype=np.float32)
                info["ind_cols"].append(c + "_isna")
    for c in feats:
        if df[c].dtype == object:
            vc = df[c].value_counts()
            if len(vc) > MAX_LEVELS:
                keep = list(vc.index[: MAX_LEVELS - 1])
                info["keep_levels"][c] = keep
                df[c] = df[c].where(df[c].isin(keep), "OTHER")
    cat_cols = [c for c in feats if df[c].dtype == object]
    num_cols = [c for c in df.columns if c not in cat_cols]      # includes the _isna indicators
    for c in cat_cols:
        info["mode_ref"][c] = df[c].mode(dropna=True).iloc[0]
    for c in num_cols:
        if c in info["ind_cols"]:
            continue                                               # indicators stay raw 0/1
        mu, sd = float(df[c].mean()), float(df[c].std(ddof=0))
        info["num_stats"][c] = (mu, sd if sd > 0 else 1.0)
        df[c] = ((df[c] - mu) / info["num_stats"][c][1]).astype("float32")
    dums = pd.get_dummies(df[cat_cols], prefix=cat_cols, dtype=np.float32)
    drop_ref = [f"{c}_{info['mode_ref'][c]}" for c in cat_cols]
    dums = dums.drop(columns=[r for r in drop_ref if r in dums.columns])
    info["dummy_cols"] = list(dums.columns)
    X = pd.concat([df[num_cols], dums], axis=1)
    info["design_cols"] = list(X.columns)
    return X, info


def transform_test(test_pdf, info):
    """Replay the TRAIN-learned design on the test frame. Unseen categorical levels
    disappear (no dummy column); levels absent from test are zero-filled."""
    feats = info["features"]
    df = _normalize(test_pdf, feats)
    for c in feats:
        na = df[c].isna()
        if c in info["impute"]:
            df[c] = df[c].fillna(info["impute"][c])
        elif na.any():    # nulls that did not exist in training: mode / train-mean fallback
            df[c] = df[c].fillna(info["mode_ref"][c] if df[c].dtype == object else 0.0)
        if c + "_isna" in info["ind_cols"]:
            df[c + "_isna"] = na.to_numpy(dtype=float)
    for c, keep in info["keep_levels"].items():
        df[c] = df[c].where(df[c].isin(keep), "OTHER")
    cat_cols = list(info["mode_ref"].keys())
    num_cols = [c for c in df.columns if c not in cat_cols]
    for c in num_cols:
        if c in info["ind_cols"]:
            continue
        mu, sd = info["num_stats"][c]
        df[c] = ((df[c] - mu) / sd).astype("float32")
    dums = pd.get_dummies(df[cat_cols], prefix=cat_cols, dtype=np.float32)
    dums = dums.reindex(columns=info["dummy_cols"], fill_value=0.0)
    keep_num = [c for c in num_cols if c in info["num_stats"] or c in info["ind_cols"]]
    X = pd.concat([df[keep_num], dums], axis=1)
    return X.reindex(columns=info["design_cols"], fill_value=0.0)


def fit_glm(X, y, w):
    """Tweedie GLM p=1.5, log link, offset log(exposure), weights=exposure — hand-rolled
    Fisher scoring (IRLS), chunked so the 980k x 67 motor design fits in driver memory
    (statsmodels' own IRLS OOM-killed the kernel). Estimator is IDENTICAL to statsmodels
    GLM: W_i = w_i*mu_i^(2-p), working response z = log(mu) + (y-mu)/mu, normal equations
    solved per iteration. Score equations sum_i w_i(y_i-mu_i)x_i = 0 mean the
    exposure-weighted claims total is matched EXACTLY on training data (assumption 9).
    Returns dict: beta (const first), bse, pvalues (normal approx), Pearson scale."""
    Xn = np.asarray(X)
    n, q = Xn.shape
    p = q + 1

    def _xk(s, e):
        out = np.empty((e - s, p), dtype=np.float64)
        out[:, 0] = 1.0
        out[:, 1:] = Xn[s:e]
        return out

    beta = np.zeros(p, dtype=np.float64)
    beta[0] = np.log(max(float((w * y).sum() / (w * w).sum()), 1e-8))  # start mu = w*exp(c)
    step, converged = 250_000, False
    for it in range(1, 101):
        eta = np.empty(n, dtype=np.float64)
        for s in range(0, n, step):
            e = min(s + step, n)
            eta[s:e] = _xk(s, e) @ beta
        mu = np.maximum(w * np.exp(np.clip(eta, -40.0, 40.0)), 1e-12)
        W = w * np.power(mu, 2.0 - POWER)          # w * mu^(2-p) = w * sqrt(mu)
        z = np.log(mu) + (y - mu) / mu             # eta = log(mu), offset included
        XtWX = np.zeros((p, p), dtype=np.float64)
        XtWz = np.zeros(p, dtype=np.float64)
        for s in range(0, n, step):
            e = min(s + step, n)
            Xk = _xk(s, e)
            Wk = W[s:e]
            XtWX += Xk.T @ (Wk[:, None] * Xk)
            XtWz += Xk.T @ (Wk * z[s:e])
        XtWX[np.diag_indices_from(XtWX)] += 1e-10
        beta_new = np.linalg.solve(XtWX, XtWz)
        delta = float(np.max(np.abs(beta_new - beta)))
        beta = beta_new
        if delta < 1e-8 * max(1.0, float(np.max(np.abs(beta)))):
            converged = True
            break
    # covariance at convergence: phi * (X'WX)^-1 with Pearson dispersion
    XtWX = np.zeros((p, p), dtype=np.float64)
    pearson = 0.0
    for s in range(0, n, step):
        e = min(s + step, n)
        Xk = _xk(s, e)
        mu_k = mu[s:e]
        Wk = w[s:e] * np.power(mu_k, 2.0 - POWER)
        XtWX += Xk.T @ (Wk[:, None] * Xk)
        pearson += float(np.sum(w[s:e] * (y[s:e] - mu_k) ** 2 / np.power(mu_k, POWER)))
    df_resid = n - p
    scale = pearson / df_resid
    cov = scale * np.linalg.solve(XtWX, np.eye(p))
    bse = np.sqrt(np.maximum(np.diag(cov), 0.0))
    pvalues = 2.0 * _norm_dist.sf(np.abs(beta) / np.maximum(bse, 1e-300))
    return dict(beta=beta, bse=bse, pvalues=pvalues, scale=float(scale),
                df_resid=df_resid, n_iter=it, converged=converged)


def glm_predict(res, X, w):
    Xn = np.asarray(X)
    eta = np.clip(res["beta"][0] + Xn @ res["beta"][1:], -40.0, 40.0)
    return np.asarray(w * np.exp(eta), dtype=float)


def tweedie_deviance(y_true, y_pred, p=POWER):
    """Mean Tweedie unit deviance (1<p<2) — the loss form the hackathon scores on."""
    y_true = np.maximum(np.asarray(y_true, dtype=float), 0.0)
    y_pred = np.maximum(np.asarray(y_pred, dtype=float), 1e-9)
    unit = 2.0 * (np.power(y_true, 2.0 - p) / ((1.0 - p) * (2.0 - p))
                  - y_true * np.power(y_pred, 1.0 - p) / (1.0 - p)
                  + np.power(y_pred, 2.0 - p) / (2.0 - p))
    return float(np.mean(unit))


def coefficient_table(res, X):
    names = ["const (base level)"] + list(X.columns)
    return pd.DataFrame({
        "term": names,
        "coef": res["beta"],
        "exp_coef_multiplier": np.exp(res["beta"]),
        "std_err": res["bse"],
        "p_value": res["pvalues"],
    }).assign(significant_5pct=lambda d: np.where(d["p_value"] < 0.05, "yes", "no"))


def run_line(line):
    """Full GLM pipeline for one line: rebuild -> design -> GroupKFold CV -> OOF check
    -> refit on 100% -> 2025 test scoring -> per-line submission CSV."""
    ref = XGB_REF[line]
    feats = LINE_CONFIG[line]["features"]

    master = rebuild_master(line)
    pdf = modeling_pandas(line, master)
    n = len(pdf)
    print(f"[{line}] {n:,} rows | claim rows {(pdf[TARGET] > 0).mean():.1%} | "
          f"dev target sum {pdf[TARGET].sum():,.0f} (mean {pdf[TARGET].mean():.2f}) | "
          f"exposure min {pdf['exposure'].min():.4f} max {pdf['exposure'].max():.2f}")

    X, info = build_design_train(pdf, feats)
    y = pdf[TARGET].to_numpy(dtype=float)
    w = pdf["exposure"].to_numpy(dtype=float)
    assert (w > 0).all(), "zero-exposure rows would break the log offset"
    print(f"[{line}] design matrix {X.shape[0]:,} x {X.shape[1]} | "
          f"{len(info['dummy_cols'])} dummies | {len(info['ind_cols'])} missing indicators")

    gkf = GroupKFold(n_splits=N_FOLDS)
    oof = np.full(n, np.nan)
    fold_rows = []
    for k, (tr, va) in enumerate(gkf.split(X, y, groups=pdf[GROUP_COL]), start=1):
        res = fit_glm(X.iloc[tr], y[tr], w[tr])
        pred = glm_predict(res, X.iloc[va], w[va])
        oof[va] = pred
        dev = tweedie_deviance(y[va], pred)
        fold_rows.append({"fold": k, "train_rows": len(tr), "val_rows": len(va),
                          "tweedie_deviance_p1.5": dev,
                          "mean_pred": float(pred.mean()), "mean_actual": float(y[va].mean())})
        print(f"[{line}] Fold {k}: deviance {dev:.4f} | mean pred {pred.mean():.2f} vs actual {y[va].mean():.2f}")
    fold_df = pd.DataFrame(fold_rows)
    print(f"[{line}] MEAN DEVIANCE {fold_df['tweedie_deviance_p1.5'].mean():.4f} "
          f"(XGBoost reference: {ref['deviance']})")

    dec = pd.qcut(pd.Series(oof), 10, labels=False, duplicates="drop") + 1
    dec_df = (pdf.assign(_oof=oof, _dec=dec).groupby("_dec")
              .agg(rows=("_oof", "size"), mean_pred=("_oof", "mean"),
                   mean_actual=(TARGET, "mean")))
    mono = (dec_df["mean_actual"].diff().dropna() >= 0).mean()
    lift = dec_df["mean_actual"].iloc[-1] / pdf[TARGET].mean()
    print(f"[{line}] decile monotonicity {mono:.0%} | top-decile lift {lift:.2f}x "
          f"(XGBoost reference: {ref['monotonicity']:.0%} / {ref['lift']}x)")

    res_full = fit_glm(X, y, w)
    coefs = coefficient_table(res_full, X)
    print(f"[{line}] full refit: {len(coefs)} terms, {(coefs['p_value'] < 0.05).sum()} significant at 5% | "
          f"dispersion (Pearson) {res_full["scale"]:,.1f}")

    test_pdf = test_pandas(line)
    X_te = transform_test(test_pdf, info)
    pred_te = glm_predict(res_full, X_te, test_pdf["exposure"].to_numpy(dtype=float))

    prefix = line[0].upper()
    assert test_pdf["row_id"].is_unique
    sub = dataframes["sample_submission"].select("row_id").toPandas()
    sub = sub.merge(pd.DataFrame({"row_id": test_pdf["row_id"],
                                   "prediction": np.round(pred_te, 2)}),
                    on="row_id", how="left")
    sub["prediction"] = sub["prediction"].fillna(DUMMY)
    sub.loc[sub["prediction"] <= 0, "prediction"] = DUMMY
    mask = sub["row_id"].str[0] == prefix
    assert len(sub) == 626_639 and mask.sum() == len(test_pdf), "submission skeleton mismatch"
    sub_path = (f"/Workspace/Users/bridget.macdonnell1@gmail.com/hackathon-project/"
                f"submissions/{line}_tweedie_glm_submission_v001.csv")
    sub.to_csv(sub_path, index=False)
    print(f"[{line}] test rows {mask.sum():,} | mean pred {sub.loc[mask, 'prediction'].mean():.2f} "
          f"vs train mean {pdf[TARGET].mean():.2f} | saved {sub_path}")

    art = dict(fold_df=fold_df, dec_df=dec_df, coefs=coefs, info=info,
               monotonicity=float(mono), lift=float(lift),
               mean_deviance=float(fold_df["tweedie_deviance_p1.5"].mean()),
               dispersion=float(res_full["scale"]),
               test_mean=float(sub.loc[mask, "prediction"].mean()),
               train_mean=float(pdf[TARGET].mean()))
    del pdf, X, master
    gc.collect()
    return art


glm_results = {}

#### Motor — Tweedie GLM
14 rating factors (top-15 XGBoost gain minus exposure). XGBoost reference: CV deviance 135.32, monotonicity 89%, top-decile lift 2.30x.

In [0]:
glm_results["motor"] = run_line("motor")
display(glm_results["motor"]["fold_df"])
display(glm_results["motor"]["coefs"].head(25))

#### Buildings — Tweedie GLM
14 rating factors (top-15 XGBoost gain minus exposure). XGBoost reference: CV deviance 140.13, monotonicity 100%, top-decile lift 2.79x.

In [0]:
glm_results["buildings"] = run_line("buildings")
display(glm_results["buildings"]["fold_df"])
display(glm_results["buildings"]["coefs"].head(25))

#### Contents — Tweedie GLM
14 rating factors (top-15 XGBoost gain minus exposure). XGBoost reference: CV deviance 70.97, monotonicity 100%, top-decile lift 2.64x.

In [0]:
glm_results["contents"] = run_line("contents")
display(glm_results["contents"]["fold_df"])
display(glm_results["contents"]["coefs"].head(25))

#### Combined v013 (GLM) submission + Excel results
Assembles M/B/C from the three GLM files above and applies the SAME per-line calibration as v012 (2026-10-21 level fix): each line's mean prediction is scaled to its DEVELOPED training mean claims/row. Because a GLM with intercept + log link + exposure offset matches the exposure-weighted claims total exactly on training data, these calibration factors should be close to 1 — measured and logged below.

In [0]:
import os

sub_dir = "/Workspace/Users/bridget.macdonnell1@gmail.com/hackathon-project/submissions"

# Developed training row-mean claims per line (measured in the 2026-10-21 rebuild runs;
# same targets v012 calibrated the XGBoost predictions to)
CAL_TARGET = {"M": 494.96, "B": 372.94, "C": 97.80}


def line_rows(path, prefix):
    df = pd.read_csv(path)
    out = df[df["row_id"].str[0] == prefix][["row_id", "prediction"]].copy()
    assert out["row_id"].is_unique
    return out


m = line_rows(os.path.join(sub_dir, "motor_tweedie_glm_submission_v001.csv"), "M")
b = line_rows(os.path.join(sub_dir, "buildings_tweedie_glm_submission_v001.csv"), "B")
c = line_rows(os.path.join(sub_dir, "contents_tweedie_glm_submission_v001.csv"), "C")

print("Per-line calibration to the developed training mean (v012 method):")
for d, p, ln in [(m, "M", "motor"), (b, "B", "buildings"), (c, "C", "contents")]:
    train_mean = glm_results[ln]["train_mean"]
    print(f"  {p} ({ln}): uncalibrated test mean {d['prediction'].mean():.2f} | "
          f"train mean {train_mean:.2f} (test/train {d['prediction'].mean()/train_mean:.3f}) | "
          f"target {CAL_TARGET[p]:.2f}")
    d["prediction"] = (d["prediction"] * CAL_TARGET[p] / d["prediction"].mean()).round(2)

v13 = pd.concat([m, b, c], ignore_index=True)
assert len(v13) == 626_639, f"row count {len(v13):,} != 626,639"
assert v13["row_id"].is_unique, "duplicate row_ids"
assert v13["prediction"].isna().sum() == 0 and (v13["prediction"] <= 0).sum() == 0

print(f"\nv013_glm: {len(v13):,} rows | by line {v13['row_id'].str[0].value_counts().to_dict()}")
for name, d in [("M (motor GLM)", m), ("B (buildings GLM)", b), ("C (contents GLM)", c)]:
    print(f"  {name}: {len(d):,} rows | mean {d['prediction'].mean():.2f} | "
          f"min {d['prediction'].min():.2f} | max {d['prediction'].max():.2f}")

v13_path = os.path.join(sub_dir, "motor_buildings_contents_submission_v013_glm.csv")
v13.to_csv(v13_path, index=False)
print(f"\nCombined GLM submission saved to: {v13_path}")

# ---- Excel results workbook: GLM vs XGBoost summary, folds, deciles, coefficients ----
excel_path = ("/Workspace/Users/bridget.macdonnell1@gmail.com/hackathon-project/"
              "Excel files/glm_tweedie_results.xlsx")
summary = pd.DataFrame([
    dict(line=ln,
         glm_deviance=glm_results[ln]["mean_deviance"], xgb_deviance=XGB_REF[ln]["deviance"],
         glm_monotonicity=glm_results[ln]["monotonicity"], xgb_monotonicity=XGB_REF[ln]["monotonicity"],
         glm_lift=glm_results[ln]["lift"], xgb_lift=XGB_REF[ln]["lift"],
         dispersion=glm_results[ln]["dispersion"])
    for ln in ["motor", "buildings", "contents"]])
display(summary)

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    summary.to_excel(writer, sheet_name="GLM_vs_XGB", index=False)
    for ln in ["motor", "buildings", "contents"]:
        pre = ln[:3]
        glm_results[ln]["fold_df"].to_excel(writer, sheet_name=f"{pre}_folds", index=False)
        glm_results[ln]["dec_df"].to_excel(writer, sheet_name=f"{pre}_deciles", index=False)
        glm_results[ln]["coefs"].to_excel(writer, sheet_name=f"{pre}_coefficients", index=False)
print(f"GLM results workbook saved to: {excel_path}")

#### Combined v014 (GLM) — calibration to the DEVELOPED 2024 level per exposure

Trend analysis (2026-10-22, AI_ASSISTANCE.md): developed pure premium per exposure trends UP ~4%/yr for buildings (severity-driven) and ~3%/yr for contents, while motor frequency is declining (~3%/yr) — the 2021-24 blended mean used for v012/v013 calibration understates the buildings/contents 2025 level and overstates motor. v014 recalibrates each line to the DEVELOPED 2024 pure premium **per exposure** (rate basis — the right target form because GLM predictions already scale with exposure). Development and exposure assumptions are identical to the rebuild cells; this cell is self-contained (reads the raw parquets and the per-line GLM v001 CSVs, needs no prior cell state).

In [0]:
import os
import numpy as np
import pandas as pd
from pyspark.sql import functions as F

# v014 — per-line calibration to the DEVELOPED 2024 pure premium PER EXPOSURE
# (rate basis) instead of the 2021-24 blended row mean used in v012/v013.
# Self-contained: reads the raw parquets + the per-line GLM v001 submission files.
data_dir = "/Workspace/Users/bridget.macdonnell1@gmail.com/hackathon-project/data"
sub_dir = "/Workspace/Users/bridget.macdonnell1@gmail.com/hackathon-project/submissions"
DUMMY = 0.01
PREFIX = {"motor": "M", "buildings": "B", "contents": "C"}


def load(name):
    return spark.read.parquet(os.path.join(data_dir, f"{name}.parquet"))


def dev_pp_2024(line):
    """Developed 2024 pure premium per exposure, identical assumptions to the
    rebuild cells: open-claim tail by peril class (2024: long x1.40, medium x1.15,
    short x1.05; closed/repudiated x1.0), IBNR 2024 (M x1.033, B x1.017, C x1.007),
    per-line exposure rule (buildings calendar-year divisor 366 for 2024, motor/
    contents fixed 365), loss year matched to period year, all-2024 policies."""
    tail = (F.when(F.col("status") != "open", F.lit(1.0))
              .when(F.col("peril").isin("bodily_injury", "subsidence"), F.lit(1.40))
              .when(F.col("peril").isin("fire", "flood"), F.lit(1.15))
              .otherwise(F.lit(1.05)))
    ibnr = {"motor": 1.033, "buildings": 1.017, "contents": 1.007}[line]
    cl = (load("claims").filter(F.year("loss_date") == 2024)
          .withColumn("net", F.col("incurred_to_date") - F.coalesce("recoveries", F.lit(0.0)))
          .withColumn("net_dev", F.col("net") * tail * F.lit(ibnr))
          .groupBy("policy_id").agg(F.sum("net_dev").alias("claims_dev")))
    div = 366.0 if line == "buildings" else 365.0
    p = (load(f"policies_{line}").filter(F.year("period_start") == 2024)
         .withColumn("exposure",
                     F.least((F.datediff("period_end", "period_start") + 1) / F.lit(div), F.lit(1.0)))
         .select("policy_id", "exposure"))
    j = p.join(cl, "policy_id", "left").withColumn("clm", F.coalesce("claims_dev", F.lit(0.0)))
    r = j.agg(F.sum("clm").alias("clm"), F.sum("exposure").alias("exp")).collect()[0]
    return float(r["clm"]) / float(r["exp"])


def test_rate(line):
    """Test-row exposure (2025: /365 for all lines) for the rate basis."""
    t = (load(f"test_{line}")
         .withColumn("exposure",
                     F.least((F.datediff("period_end", "period_start") + 1) / F.lit(365.0), F.lit(1.0)))
         .select("row_id", "exposure")).toPandas()
    assert t["row_id"].is_unique and (t["exposure"] > 0).all()
    return t


parts = []
print("v014: per-line calibration to the DEVELOPED 2024 pure premium per exposure")
for line in ["motor", "buildings", "contents"]:
    target = dev_pp_2024(line)
    sub = pd.read_csv(f"{sub_dir}/{line}_tweedie_glm_submission_v001.csv")
    sub = (sub[sub["row_id"].str[0] == PREFIX[line]]
           .merge(test_rate(line), on="row_id", how="left"))
    assert sub["exposure"].notna().all()
    rate_mean = float((sub["prediction"] / sub["exposure"]).mean())
    factor = target / rate_mean
    sub["prediction"] = (sub["prediction"] * factor).round(2)
    sub.loc[sub["prediction"] <= 0, "prediction"] = DUMMY
    print(f"  {PREFIX[line]} ({line}): 2024 dev pp/exp {target:,.1f} | test rate mean {rate_mean:,.1f} "
          f"| factor x{factor:,.3f} -> row mean {sub['prediction'].mean():,.2f}")
    parts.append(sub[["row_id", "prediction"]])

skel = load("sample_submission").select("row_id").toPandas()
v14 = skel.merge(pd.concat(parts).drop_duplicates("row_id"), on="row_id", how="left")
assert len(v14) == 626_639 and v14["row_id"].is_unique
assert v14["prediction"].notna().all() and (v14["prediction"] > 0).all()
out_path = f"{sub_dir}/motor_buildings_contents_submission_v014_glm_2024level.csv"
v14.to_csv(out_path, index=False)
print(f"\nv014 saved: {out_path}")
for k, g in v14.assign(line=v14["row_id"].str[0]).groupby("line"):
    print(f"  {k}: {len(g):,} rows | mean {g['prediction'].mean():,.2f} | "
          f"min {g['prediction'].min():.2f} | max {g['prediction'].max():,.2f}")

#### Full-feature GLM+XGBoost blend — motor, buildings & contents

For each line: rebuild the master, run a full-feature XGBoost (40–50 features, native categoricals) and the 14-feature GLM on the SAME GroupKFold splits, search for the optimal blend weight, refit on 100% of training data, and save a blended submission.

In [0]:
%pip install xgboost -q
import xgboost as xgb

# Full XGBoost feature lists from the per-line XGBoost notebooks (2026-10-21/22)
XGB_FULL_FEATURES = {
    "motor": [
        "Year", "exposure",
        "driver_age", "years_licensed", "ncb_years", "vehicle_value_group", "vehicle_age",
        "make_model", "engine_size_cc", "body_type", "transmission", "seats", "fuel_type",
        "driver_assist", "cover_type", "annual_mileage", "overnight_parking", "vehicle_use",
        "named_young_driver", "telematics_opt_in", "telematics_score",
        "voluntary_excess", "compulsory_excess",
        "channel", "payment_method", "missed_payments_12m", "new_business", "renewal_count",
        "days_quote_to_inception",
        "occupation", "marital_status", "credit_band", "prior_claims_5y", "licence_points",
        "branch_id", "marketing_segment",
        "region", "urban_density",
        "hail_days",
    ],
    "buildings": [
        "Year", "exposure",
        "policyholder_age", "sum_insured", "accidental_damage_cover", "construction_type",
        "occupancy", "days_unoccupied", "smoke_alarms", "mortgage",
        "voluntary_excess", "compulsory_excess",
        "channel", "payment_method", "missed_payments_12m", "new_business", "renewal_count",
        "days_quote_to_inception",
        "occupation", "marital_status", "household_size", "credit_band", "prior_claims_5y",
        "licence_points", "years_at_address",
        "branch_id", "marketing_segment", "region", "urban_density", "flood_zone",
        "distance_to_river_km", "elevation_m", "soil_type", "year_built", "property_type",
        "roof_type", "basement", "trees_within_5m", "listed_building", "bedrooms",
        "energy_rating", "solar_panels",
        "hail_days", "wind_exposure_index", "freezing_days", "dry_summer_index",
        "wildfire_index", "rainfall_mm", "mean_temperature_c",
    ],
    "contents": [
        "Year", "exposure",
        "policyholder_age", "sum_insured", "tenure", "security_level", "high_value_items",
        "accidental_damage_cover", "works_from_home", "lodgers", "bicycle_cover",
        "voluntary_excess", "compulsory_excess",
        "channel", "payment_method", "missed_payments_12m", "new_business", "renewal_count",
        "days_quote_to_inception",
        "occupation", "marital_status", "household_size", "credit_band", "prior_claims_5y",
        "licence_points", "years_at_address",
        "branch_id", "marketing_segment", "region", "urban_density", "flood_zone",
        "distance_to_river_km", "elevation_m", "soil_type", "year_built", "property_type",
        "roof_type", "basement", "trees_within_5m", "listed_building", "bedrooms",
        "energy_rating", "solar_panels",
        "hail_days", "wind_exposure_index", "freezing_days", "dry_summer_index",
        "wildfire_index", "rainfall_mm", "mean_temperature_c",
    ],
}

XGB_BLEND_PARAMS = dict(
    objective="reg:tweedie",
    tweedie_variance_power=POWER,
    tree_method="hist",
    enable_categorical=True,
    max_depth=6,
    learning_rate=0.05,
    n_estimators=300,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
)


def fit_glm_ridge(X, y, w, alpha=0.0):
    """Tweedie GLM p=1.5, log link, offset log(exposure), weights=exposure, with L2
    penalty alpha on all coefficients EXCEPT the intercept. Same chunked IRLS as
    fit_glm; the penalty enters as XtWX += alpha * diag(0,1,...,1)."""
    Xn = np.asarray(X, dtype=np.float64)
    n, q = Xn.shape
    p = q + 1
    pen_diag = np.zeros(p, dtype=np.float64)
    pen_diag[1:] = alpha

    def _xk(s, e):
        out = np.empty((e - s, p), dtype=np.float64)
        out[:, 0] = 1.0
        out[:, 1:] = Xn[s:e]
        return out

    beta = np.zeros(p, dtype=np.float64)
    beta[0] = np.log(max(float((w * y).sum() / (w * w).sum()), 1e-8))
    step = 250_000
    for it in range(1, 101):
        eta = np.empty(n, dtype=np.float64)
        for s in range(0, n, step):
            e = min(s + step, n)
            eta[s:e] = _xk(s, e) @ beta
        mu = np.maximum(w * np.exp(np.clip(eta, -40.0, 40.0)), 1e-12)
        W = w * np.power(mu, 2.0 - POWER)
        z = np.log(mu) + (y - mu) / mu
        XtWX = np.zeros((p, p), dtype=np.float64)
        XtWz = np.zeros(p, dtype=np.float64)
        for s in range(0, n, step):
            e = min(s + step, n)
            Xk = _xk(s, e)
            Wk = W[s:e]
            XtWX += Xk.T @ (Wk[:, None] * Xk)
            XtWz += Xk.T @ (Wk * z[s:e])
        XtWX[np.diag_indices_from(XtWX)] += pen_diag + 1e-10
        beta_new = np.linalg.solve(XtWX, XtWz)
        delta = float(np.max(np.abs(beta_new - beta)))
        beta = beta_new
        if delta < 1e-8 * max(1.0, float(np.max(np.abs(beta)))):
            break
    return dict(beta=beta, n_iter=it)


def run_blend(line):
    """Full-feature GLM+XGBoost blend for one line: rebuild → full-feature XGBoost CV
    + 14-feature GLM CV (same GroupKFold splits) → blend search → refit → test → save."""
    feats_xgb = XGB_FULL_FEATURES[line]
    feats_glm = LINE_CONFIG[line]["features"]
    prefix = line[0].upper()

    # 1. Rebuild master + derive Year + recompute exposure
    master = rebuild_master(line)
    master = (master
        .withColumn("Year", F.year("period_start"))
        .withColumn("exposure", exposure_expr(line))
        .withColumn(TARGET, F.coalesce(F.col(TARGET), F.lit(0.0))))
    if line == "buildings":
        master = master.withColumn("construction_type", F.lower(F.trim(F.col("construction_type"))))

    # 2. Convert to pandas ONCE (same row order for both models)
    needed = list(set(feats_xgb) | set(feats_glm) | {"row_id", "policy_id", GROUP_COL, "exposure", TARGET})
    available = [c for c in master.columns if c in needed]
    full_pdf = master.select(*available).toPandas()
    n = len(full_pdf)

    missing = [c for c in feats_xgb if c not in full_pdf.columns]
    if missing:
        print(f"  WARNING: {line} XGBoost features missing: {missing}")
        feats_xgb = [c for c in feats_xgb if c in full_pdf.columns]

    y = full_pdf[TARGET].to_numpy(dtype=float)
    w = full_pdf["exposure"].to_numpy(dtype=float)
    groups = full_pdf[GROUP_COL].to_numpy()

    # 3. XGBoost matrix (native categoricals)
    X_xgb = full_pdf[feats_xgb].copy()
    bool_cols = [c for c in feats_xgb if X_xgb[c].dtype == bool]
    for c in bool_cols:
        X_xgb[c] = X_xgb[c].astype("int8")
    cat_cols = [c for c in feats_xgb if str(X_xgb[c].dtype) == "object"]
    for c in cat_cols:
        X_xgb[c] = X_xgb[c].apply(lambda v: str(v) if pd.notna(v) else None)
        X_xgb[c] = pd.Categorical(X_xgb[c])

    # 4. GLM design matrix (14 features, one-hot encoded)
    X_glm, info = build_design_train(full_pdf, feats_glm)

    print(f"[{line}] {n:,} rows | XGB {len(feats_xgb)} feats ({len(cat_cols)} cat, {len(bool_cols)} bool) | "
          f"GLM {len(feats_glm)} feats | claims {(y > 0).mean():.1%}")

    # 5. GroupKFold (same splits for both)
    gkf = GroupKFold(n_splits=N_FOLDS)
    folds = list(gkf.split(X_xgb, y, groups=groups))

    # 6. XGBoost CV
    xgb_oof = np.full(n, np.nan)
    for k, (tr, va) in enumerate(folds, start=1):
        model = xgb.XGBRegressor(**XGB_BLEND_PARAMS)
        model.fit(X_xgb.iloc[tr], y[tr], sample_weight=w[tr])
        xgb_oof[va] = model.predict(X_xgb.iloc[va])
        print(f"  [{line}] XGB Fold {k}: dev {tweedie_deviance(y[va], xgb_oof[va]):.4f} | "
              f"pred {xgb_oof[va].mean():.1f} vs actual {y[va].mean():.1f}")
    xgb_dev = tweedie_deviance(y, xgb_oof)

    # 7. GLM CV (unpenalized)
    glm_oof = np.full(n, np.nan)
    for k, (tr, va) in enumerate(folds, start=1):
        res = fit_glm_ridge(X_glm.iloc[tr], y[tr], w[tr], alpha=0.0)
        glm_oof[va] = glm_predict(res, X_glm.iloc[va], w[va])
    glm_dev = tweedie_deviance(y, glm_oof)

    # 8. Blend search
    weights = np.arange(0.0, 1.05, 0.05)
    blend_rows = []
    for wgt in weights:
        dev = tweedie_deviance(y, wgt * glm_oof + (1.0 - wgt) * xgb_oof)
        blend_rows.append({"glm_weight": round(float(wgt), 2), "xgb_weight": round(float(1.0 - wgt), 2), "deviance": dev})
    blend_df = pd.DataFrame(blend_rows)
    best_idx = blend_df["deviance"].idxmin()
    best_w = blend_df.loc[best_idx, "glm_weight"]

    print(f"  [{line}] GLM {glm_dev:.4f} | XGB {xgb_dev:.4f} | "
          f"Best blend w_glm={best_w:.2f} → {blend_df.loc[best_idx, 'deviance']:.4f}")

    # 9. Refit both on 100%
    glm_final = fit_glm_ridge(X_glm, y, w, alpha=0.0)
    xgb_final = xgb.XGBRegressor(**XGB_BLEND_PARAMS)
    xgb_final.fit(X_xgb, y, sample_weight=w)

    # 10. Build test master
    wx = (dataframes["weather"]
          .withColumnRenamed("address_id", "_wx_address_id")
          .withColumnRenamed("year", "_wx_year"))
    test_master = (dataframes[f"test_{line}"]
        .join(wx, (F.col("address_id") == F.col("_wx_address_id")) &
              (F.year(F.col("period_start")) == F.col("_wx_year")), "left")
        .drop("_wx_address_id", "_wx_year")
        .join(dataframes["customers"], "customer_id", "left")
        .join(dataframes["addresses"], "address_id", "left")
        .withColumn("Year", F.year("period_start"))
        .withColumn("exposure", exposure_expr(line)))
    if line == "buildings":
        test_master = test_master.withColumn("construction_type", F.lower(F.trim(F.col("construction_type"))))

    test_pdf = test_master.select("row_id", *feats_xgb).toPandas()

    # 11. XGBoost test matrix (align categoricals with training)
    X_test_xgb = test_pdf[feats_xgb].copy()
    for c in bool_cols:
        X_test_xgb[c] = X_test_xgb[c].astype("int8")
    for c in cat_cols:
        cats = X_xgb[c].cat.categories
        X_test_xgb[c] = test_pdf[c].apply(lambda v: str(v) if pd.notna(v) else None)
        X_test_xgb[c] = pd.Categorical(X_test_xgb[c], categories=cats)

    # 12. GLM test matrix
    test_glm_pdf = test_master.select("row_id", *feats_glm).toPandas()
    X_test_glm = transform_test(test_glm_pdf, info)
    w_te = test_master.select("exposure").toPandas()["exposure"].to_numpy(dtype=float)

    # 13. Score test
    glm_test = glm_predict(glm_final, X_test_glm, w_te)
    xgb_test = xgb_final.predict(X_test_xgb)
    blended_test = np.maximum(np.round(best_w * glm_test + (1.0 - best_w) * xgb_test, 2), DUMMY)

    # 14. Save submission
    sub = dataframes["sample_submission"].select("row_id").toPandas()
    pred_map = pd.DataFrame({"row_id": test_pdf["row_id"], "prediction": blended_test})
    sub = sub.merge(pred_map, on="row_id", how="left")
    sub["prediction"] = sub["prediction"].fillna(DUMMY)
    sub.loc[sub["prediction"] <= 0, "prediction"] = DUMMY

    mask = sub["row_id"].str[0] == prefix
    sub_path = f"/Workspace/Users/bridget.macdonnell1@gmail.com/hackathon-project/submissions/{line}_blend_full_xgb_submission_v001.csv"
    sub.to_csv(sub_path, index=False)

    print(f"  [{line}] Saved: {sub_path}")
    print(f"    {prefix} rows: {mask.sum():,} | mean {sub.loc[mask, 'prediction'].mean():.2f} | "
          f"min {sub.loc[mask, 'prediction'].min():.2f} | max {sub.loc[mask, 'prediction'].max():.2f}")

    return dict(line=line, glm_dev=glm_dev, xgb_dev=xgb_dev,
                blend_dev=blend_df.loc[best_idx, "deviance"], best_w=best_w,
                blend_df=blend_df, sub_path=sub_path)


blend_results = {}
print("run_blend() defined — call run_blend('motor' / 'buildings' / 'contents')")

In [0]:
# XGBoost hyperparameter tuning for all 3 lines — tests 10 configs per line using GroupKFold CV deviance.
# Motor reverted to v2 features (39 XGB, 15 GLM) as the baseline.
# Grid focuses on overfitting controls: max_depth, reg_lambda, colsample_bytree, min_child_weight.

import time

TUNE_GRID = [
    {"max_depth": 6, "n_estimators": 300, "learning_rate": 0.05, "reg_lambda": 1.0, "colsample_bytree": 0.8, "subsample": 0.8, "min_child_weight": 5},   # 1 baseline
    {"max_depth": 5, "n_estimators": 500, "learning_rate": 0.05, "reg_lambda": 3.0, "colsample_bytree": 0.7, "subsample": 0.8, "min_child_weight": 10},  # 2 regularized
    {"max_depth": 4, "n_estimators": 500, "learning_rate": 0.05, "reg_lambda": 3.0, "colsample_bytree": 0.7, "subsample": 0.8, "min_child_weight": 10},  # 3 shallow + regularized
    {"max_depth": 6, "n_estimators": 800, "learning_rate": 0.03, "reg_lambda": 5.0, "colsample_bytree": 0.7, "subsample": 0.8, "min_child_weight": 10},  # 4 deep + heavy reg
]

best_params = {}

for line in ["motor", "buildings", "contents"]:
    feats_xgb = XGB_FULL_FEATURES[line]

    # Rebuild master + derive Year + recompute exposure
    master = rebuild_master(line)
    master = (master
        .withColumn("Year", F.year("period_start"))
        .withColumn("exposure", exposure_expr(line))
        .withColumn(TARGET, F.coalesce(F.col(TARGET), F.lit(0.0))))
    if line == "buildings":
        master = master.withColumn("construction_type", F.lower(F.trim(F.col("construction_type"))))

    # Convert to pandas
    needed = list(set(feats_xgb) | {"row_id", "policy_id", GROUP_COL, "exposure", TARGET})
    full_pdf = master.select(*[c for c in master.columns if c in needed]).toPandas()
    n = len(full_pdf)

    y = full_pdf[TARGET].to_numpy(dtype=float)
    w = full_pdf["exposure"].to_numpy(dtype=float)
    groups = full_pdf[GROUP_COL].to_numpy()

    # XGBoost matrix (native categoricals)
    X_xgb = full_pdf[feats_xgb].copy()
    bool_cols = [c for c in feats_xgb if X_xgb[c].dtype == bool]
    for c in bool_cols:
        X_xgb[c] = X_xgb[c].astype("int8")
    cat_cols = [c for c in feats_xgb if str(X_xgb[c].dtype) == "object"]
    for c in cat_cols:
        X_xgb[c] = X_xgb[c].apply(lambda v: str(v) if pd.notna(v) else None)
        X_xgb[c] = pd.Categorical(X_xgb[c])

    # GroupKFold splits (same for all configs)
    gkf = GroupKFold(n_splits=N_FOLDS)
    folds = list(gkf.split(X_xgb, y, groups=groups))

    print(f"\n{'='*70}")
    print(f"[{line}] {n:,} rows | {len(feats_xgb)} XGB feats | tuning {len(TUNE_GRID)} configs")
    print(f"{'='*70}")

    line_results = []
    for ci, params in enumerate(TUNE_GRID, start=1):
        t0 = time.time()
        xgb_params = dict(
            objective="reg:tweedie",
            tweedie_variance_power=POWER,
            tree_method="hist",
            enable_categorical=True,
            random_state=42,
            n_jobs=-1,
            **params,
        )
        fold_devs = []
        for k, (tr, va) in enumerate(folds):
            model = xgb.XGBRegressor(**xgb_params)
            model.fit(X_xgb.iloc[tr], y[tr], sample_weight=w[tr])
            pred = model.predict(X_xgb.iloc[va])
            fold_devs.append(tweedie_deviance(y[va], pred))

        mean_dev = np.mean(fold_devs)
        std_dev = np.std(fold_devs)
        elapsed = time.time() - t0
        tag = " <-- BASELINE" if ci == 1 else ""
        print(f"  [{ci:2d}] d={params['max_depth']} n={params['n_estimators']:4d} lr={params['learning_rate']:.2f} "
              f"reg_lambda={params['reg_lambda']:.0f} col={params['colsample_bytree']:.1f} mcw={params['min_child_weight']:2d} | "
              f"dev {mean_dev:.4f} +/- {std_dev:.4f} ({elapsed:.0f}s){tag}")
        line_results.append((mean_dev, params, fold_devs))

    best = min(line_results, key=lambda x: x[0])
    best_params[line] = best[1]
    improvement = (line_results[0][0] - best[0]) / line_results[0][0] * 100
    print(f"\n  >>> [{line}] BEST: dev {best[0]:.4f} (baseline {line_results[0][0]:.4f}, improvement {improvement:.1f}%)")
    print(f"      d={best[1]['max_depth']} n={best[1]['n_estimators']} lr={best[1]['learning_rate']} "
          f"reg_lambda={best[1]['reg_lambda']} col={best[1]['colsample_bytree']} mcw={best[1]['min_child_weight']}")

    # Free memory
    del full_pdf, X_xgb, y, w, groups
    gc.collect()

print(f"\n{'='*70}")
print("BEST PARAMS PER LINE:")
for line, params in best_params.items():
    print(f"  {line:10s}: max_depth={params['max_depth']} n_est={params['n_estimators']} "
          f"lr={params['learning_rate']} reg_lambda={params['reg_lambda']} col={params['colsample_bytree']} mcw={params['min_child_weight']}")

In [0]:
blend_results["motor"] = run_blend("motor")
display(blend_results["motor"]["blend_df"])

In [0]:
blend_results["buildings"] = run_blend("buildings")
display(blend_results["buildings"]["blend_df"])

In [0]:
blend_results["contents"] = run_blend("contents")
display(blend_results["contents"]["blend_df"])

In [0]:
# Combine the three per-line blend CSVs (each has all 626,639 rows with dummy 0.01
# for other lines) into a single submission: take the max prediction per row_id so
# the real (non-dummy) value from each line's blend file is selected.
import pandas as pd

sub_dir = "/Workspace/Users/bridget.macdonnell1@gmail.com/hackathon-project/submissions"
_per_line = [pd.read_csv(f"{sub_dir}/{line}_blend_full_xgb_submission_v001.csv").set_index('row_id')['prediction']
             for line in ['motor', 'buildings', 'contents']]
combined = pd.concat(_per_line, axis=1).max(axis=1).reset_index()
combined.columns = ['row_id', 'prediction']

out = f"{sub_dir}/motor_buildings_contents_blend_full_xgb_v001.csv"
combined.to_csv(out, index=False)

for prefix, name in [('M', 'motor'), ('B', 'buildings'), ('C', 'contents')]:
    sub = combined[combined['row_id'].str.startswith(prefix)]
    print(f"{name:10s}: {len(sub):>7,} rows | mean {sub['prediction'].mean():>8.2f} | "
          f"min {sub['prediction'].min():>6.2f} | max {sub['prediction'].max():>10.2f}")
print(f"\nTotal: {len(combined):,} rows | saved: {out}")

In [0]:
# Claims-rate scaling with 2025 trend: calibrate test predictions so each line's
# predicted claims rate (claims / exposure) matches the 2025-extrapolated training
# claims rate.  This is a PRICING exercise (full reprice), so we scale by exposure, not
# premium.  Premium is NOT used — it's a pricing output, not an input.
#
# Two corrections in one factor:
#   1. LEVEL: models systematically under-predict (blend OOF < actual)
#   2. TREND: training is 2021-2024, test is 2025.  XGBoost has Year as a feature but
#      trees can't extrapolate beyond 2024; the GLM has no Year feature at all.
#      So neither model captures a 2025 trend.  We extrapolate the training claims-rate
#      trend (geometric mean YoY) from 2024 to 2025.
#
# factor = 2025_expected_rate / model_predicted_rate
# where 2025_expected_rate = 2024_rate * (1 + geom_YoY_growth)
import pandas as pd
import numpy as np
from pyspark.sql import functions as F

sub_dir = "/Workspace/Users/bridget.macdonnell1@gmail.com/hackathon-project/submissions"

# 1. Training claims rate by year per line (using rebuild_master + exposure_expr)
train_rates = {}
for line in ["motor", "buildings", "contents"]:
    master = rebuild_master(line)
    master = (master
        .withColumn("exposure", exposure_expr(line))
        .withColumn("Year", F.year("period_start"))
        .withColumn(TARGET, F.coalesce(F.col(TARGET), F.lit(0.0))))
    yr = (master.groupBy("Year")
          .agg(F.sum(TARGET).alias("claims"), F.sum("exposure").alias("exp"))
          .orderBy("Year")).toPandas()
    yr["rate"] = yr["claims"] / yr["exp"]
    rates = yr["rate"].values
    # Geometric mean YoY growth
    yoy = [rates[i+1] / rates[i] - 1 for i in range(len(rates) - 1)]
    geom_yoy = float(np.exp(np.mean(np.log(1 + np.array(yoy)))) - 1)
    rate_2024 = float(rates[-1])
    rate_2025 = rate_2024 * (1.0 + geom_yoy)
    train_rates[line] = dict(rates=rates, geom_yoy=geom_yoy, rate_2024=rate_2024, rate_2025=rate_2025)
    print(f"[{line}] rates by year: {', '.join(f'{int(y)}:{r:.1f}' for y, r in zip(yr['Year'], rates))} "
          f"| geom YoY {geom_yoy*100:+.2f}% | 2025 rate {rate_2025:.1f}")

# 2. Test exposure totals per line
print()
test_exp = {}
for line in ["motor", "buildings", "contents"]:
    tdf = dataframes[f"test_{line}"]
    tdf = tdf.withColumn("exposure", exposure_expr(line))
    test_exp[line] = float(tdf.agg(F.sum("exposure").alias("e")).toPandas()["e"].iloc[0])

# 3. Scale per-line blend predictions and combine
combined_parts = []
for line in ["motor", "buildings", "contents"]:
    prefix = line[0].upper()
    df = pd.read_csv(f"{sub_dir}/{line}_blend_full_xgb_submission_v001.csv")
    mask = df["row_id"].str.startswith(prefix)
    pred_total = df.loc[mask, "prediction"].sum()
    pred_rate = pred_total / test_exp[line]

    # Scale to 2025-extrapolated claims rate
    target_rate = train_rates[line]["rate_2025"]
    scale = target_rate / pred_rate

    df.loc[mask, "prediction"] = (df.loc[mask, "prediction"] * scale).round(2)
    df.loc[~mask, "prediction"] = DUMMY

    sub = df.loc[mask]
    print(f"{line:10s}: pred rate {pred_rate:.1f} → {target_rate:.1f} | scale {scale:.4f} | "
          f"mean {sub['prediction'].mean():>8.2f} | min {sub['prediction'].min():>6.2f} | "
          f"max {sub['prediction'].max():>10.2f}")
    combined_parts.append(df.set_index("row_id")["prediction"])

# 4. Combine and save
combined = pd.concat(combined_parts, axis=1).max(axis=1).reset_index()
combined.columns = ["row_id", "prediction"]

out = f"{sub_dir}/motor_buildings_contents_blend_scaled_v001.csv"
combined.to_csv(out, index=False)
print(f"\nTotal: {len(combined):,} rows | saved: {out}")

In [0]:
# Conservative scaling: exclude 2024 from trend estimation.
# 2024 claims are the least developed (most open claims, highest tail factors).
# The motor 2024 claims rate dropped 18% vs 2023 — likely underdevelopment, not a real
# trend reversal.  We compute the trend from 2021-2023 only (stable, developed years)
# and apply ONE year of that trend to the 2024 rate to get 2025.
#
# This gives motor an upward trend (+4.85%) instead of the full-period downward
# trend (-3.53%).  Buildings and contents are barely affected.
import pandas as pd
import numpy as np

sub_dir = "/Workspace/Users/bridget.macdonnell1@gmail.com/hackathon-project/submissions"

# Reuse train_rates and test_exp from cell 23 (already computed)
# Compute ex-2024 trend: geometric mean YoY using only 2021->2022 and 2022->2023
combined_parts = []
for line in ["motor", "buildings", "contents"]:
    rates = train_rates[line]["rates"]          # [2021, 2022, 2023, 2024]
    yoy_ex = [rates[i+1] / rates[i] - 1 for i in range(len(rates) - 2)]  # skip last (2023->2024)
    geom_ex = float(np.exp(np.mean(np.log(1 + np.array(yoy_ex)))) - 1)
    rate_2024 = float(rates[-1])
    rate_2025 = rate_2024 * (1.0 + geom_ex)

    prefix = line[0].upper()
    df = pd.read_csv(f"{sub_dir}/{line}_blend_full_xgb_submission_v001.csv")
    mask = df["row_id"].str.startswith(prefix)
    pred_total = df.loc[mask, "prediction"].sum()
    pred_rate = pred_total / test_exp[line]
    scale = rate_2025 / pred_rate

    df.loc[mask, "prediction"] = (df.loc[mask, "prediction"] * scale).round(2)
    df.loc[~mask, "prediction"] = DUMMY

    sub = df.loc[mask]
    print(f"{line:10s}: ex-2024 YoY {geom_ex*100:+.2f}% | 2024 rate {rate_2024:.1f} -> 2025 {rate_2025:.1f} | "
          f"scale {scale:.4f} | mean {sub['prediction'].mean():>8.2f} | "
          f"min {sub['prediction'].min():>6.2f} | max {sub['prediction'].max():>10.2f}")
    combined_parts.append(df.set_index("row_id")["prediction"])

combined = pd.concat(combined_parts, axis=1).max(axis=1).reset_index()
combined.columns = ["row_id", "prediction"]

out = f"{sub_dir}/motor_buildings_contents_blend_scaled_conservative_v001.csv"
combined.to_csv(out, index=False)
print(f"\nTotal: {len(combined):,} rows | saved: {out}")